# **💁🏻🗨️💁🏻‍♂️대화 요약 SOLAR API code**
> **Dialogue Summarization** 경진대회에 오신 여러분 환영합니다! 🎉    
> 본 자료에서는 Solar Chat API를 이용하여 대화 요약 대회를 풀어봅니다.     

## ⚙️ 데이터 및 환경설정

### 1) 필요한 라이브러리 설치

- 필요한 라이브러리를 설치한 후 불러옵니다.

In [28]:
!pip install openai scikit-learn

In [29]:
import pandas as pd
import os
import re
import json
import time
import numpy as np
from tqdm import tqdm
from rouge import Rouge # 모델의 성능을 평가하기 위한 라이브러리입니다.
from openai import OpenAI # openai==1.2.0

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print('라이브러리 로드 완료')


라이브러리 로드 완료


### 2) Solar Chat API Client 생성하기
- 앞으로 Solar Chat API를 사용하기 위해 Client를 생성합니다.

In [30]:
UPSTAGE_API_KEY = "up_opnjqOc0T694rx1It5hDPhfZNCiW5" # upstage.ai에서 발급받은 API KEY를 입력해주세요.

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1/solar"
)

### 3) 데이터 불러오기
- 실험에서 쓰일 데이터를 load합니다.

In [31]:
# 데이터 경로를 지정해줍니다.
DATA_PATH   = "/data/ephemeral/home/data/"
RESULT_PATH = "/data/ephemeral/home/code/prediction/"

In [32]:
# train data의 구조와 내용을 확인합니다.
train_df = pd.read_csv(os.path.join(DATA_PATH,'train.csv'))
train_df.tail()


,fname,dialogue,summary,topic
12452,train_12455,#Person1#: 안녕하세요. 혹시 맨체스터에서 오신 Mr. Green 맞으신가요...,Tan Ling은 흰머리와 수염이 특징인 Mr. Green을 맞이하여 호텔로 안내합...,호텔 안내
12453,train_12456,"#Person1#: Mister Ewing이 우리 회의장에 4시에 오라고 했지, 맞...",#Person1#과 #Person2#는 Mister Ewing의 요청에 따라 회의장...,회의 준비
12454,train_12457,#Person1#: 오늘 어떻게 도와드릴까요?\n#Person2#: 차를 빌리고 싶...,#Person2#는 #Person1#의 도움으로 5일 동안 소형차를 대여합니다.,차량 대여
12455,train_12458,#Person1#: 너 오늘 좀 기분 안 좋아 보인다? 무슨 일 있어?\n#Pers...,#Person2#의 어머니가 직장을 잃으셨다. #Person2#는 어머니가 우울해하...,실직과 대처
12456,train_12459,"#Person1#: 엄마, 나 다음 주 토요일에 이모부네 가족 보러 가는데, 오늘 ...",#Person1#은 다음 주 토요일에 이모부네 가족을 방문하기 위해 짐을 싸야 하는...,가족 방문 준비


In [33]:
# validation data의 구조와 내용을 확인합니다.
val_df = pd.read_csv(os.path.join(DATA_PATH,'dev.csv'))
val_df.tail()

,fname,dialogue,summary,topic
494,dev_495,#Person1#: 새해가 되니까 나도 새 출발을 하기로 했어.\n#Person2#...,#Person1#은 새해에 담배를 끊고 커밍아웃 하기로 결심했습니다. #Person...,새해 결심
495,dev_496,#Person1#: 너 Joe랑 결혼했지?\n#Person2#: Joe? 무슨 말이...,"#Person1#은 #Person2#가 Joe와 결혼했다고 생각하지만, #Perso...",사랑과 결혼 오해
496,dev_497,"#Person1#: 어떻게 도와드릴까요, 아줌마?\n#Person2#: 제 차에서 ...","#Person2#의 차에서 소리가 나며, 브레이크 수리가 필요한 상황입니다. #Pe...",차량 소음 및 수리
497,dev_498,"#Person1#: 여보세요, 아마존 고객 서비스입니다. 어떻게 도와드릴까요?\n#...",#Person2#가 아마존 고객 서비스에 전화하여 아마존에서 구매한 책에 53페이지...,책 페이지 누락
498,dev_499,#Person1#: 벌써 여름이 다가오다니 믿기지 않아. \n#Person2#: 맞...,"#Person2#는 여름방학 동안 파티에서 일하는 회사에서 일하며, 주로 음식 준비...",여름방학 일자리


In [34]:
# test data의 구조와 내용을 확인합니다.
test_df = pd.read_csv(os.path.join(DATA_PATH,'test.csv'))
test_df.tail()

,fname,dialogue
494,test_495,"#Person1#: 얘, Charlie, 학교 끝나고 우리 집에 와서 나랑 비디오 ..."
495,test_496,#Person1#: 어떻게 시골 음악에 관심을 갖게 되었어요?\n#Person2#:...
496,test_497,"#Person1#: 저기, Alice. 여기는 처음 와봤어요. 어떻게 기계를 사용하..."
497,test_498,#Person1#: Matthew? 안녕! \n#Person2#: Steve! 진짜...
498,test_499,"#Person1#: 어, Betsy, 좋은 소식 들었어?\n#Person2#: 아니..."


In [35]:
print(f"train: {len(train_df)}개")
print(f"val  : {len(val_df)}개")
print(f"test : {len(test_df)}개")

train: 12457개
val  : 499개
test : 499개


In [36]:
print(f'train 컬럼: {train_df.columns.tolist()}')
print(f'test  컬럼: {test_df.columns.tolist()}')  # topic 없음 확인

train 컬럼: ['fname', 'dialogue', 'summary', 'topic']
test  컬럼: ['fname', 'dialogue']


## 1. 평가지표 & TF-IDF Topic 인덱스
> **핵심**: train topic과 predicted topic을 TF-IDF 유사도로 매칭해서 few-shot 품질 향상

In [37]:
# 모델 성능에 대한 평가 지표를 정의합니다. 본 대회에서는 ROUGE 점수를 통해 모델의 성능을 평가합니다.
rouge = Rouge()

def compute_metrics(pred, gold):
    """ROUGE-1/2/L F1 평균 반환"""
    try:
        results = rouge.get_scores(str(pred), str(gold), avg=True)
        r1 = results['rouge-1']['f']
        r2 = results['rouge-2']['f']
        rl = results['rouge-l']['f']
        return {'rouge-1': r1, 'rouge-2': r2, 'rouge-l': rl, 'avg': (r1+r2+rl)/3}
    except:
        return {'rouge-1': 0, 'rouge-2': 0, 'rouge-l': 0, 'avg': 0}

In [38]:
# train topic TF-IDF 인덱스 구축 (한 번만 실행)
def build_topic_index(df):
    """train topic TF-IDF 인덱스 사전 구축"""
    topics = df['topic'].fillna('').tolist()
    vec = TfidfVectorizer(analyzer='char', ngram_range=(1, 3))
    mat = vec.fit_transform(topics)
    return vec, mat

vectorizer, topic_matrix = build_topic_index(train_df)
print(f"Topic 인텍스 구축 완료 : {len(train_df)}개")

Topic 인텍스 구축 완료 : 12457개


## 2. 핵심 함수 정의

### 2-1. topic 예측 함수
> test.csv에는 topic 컬럼이 없으므로 Solar API로 예측  
> train topic 샘플 20개를 힌트로 제공해서 예측 정확도 향상

In [39]:
def predict_topic(dialogue, max_retries=3):
    """대화에서 topic 예측 (test.csv에 topic 없으므로 Solar로 예측)"""
    """train의 실제 topic 목록을 힌트로 줘서 예측 정확도 향상"""

    # train topic 샘플 20개를 힌트로 제공
    topic_samples = train_df['topic'].dropna().unique()
    topic_hint = ', '.join(
        np.random.choice(topic_samples, min(20, len(topic_samples)), replace=False))

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
            model="solar-1-mini-chat",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "대화를 읽고 핵심 주제를 2~4단어로 짧게 답하세요. \n"
                        f"참고: 아래와 같은 형식의 주제를 사용하세요.\n"
                        f"예시 주제들: {topic_hint}\n"
                        "반드시 주제만 답하고 다른 말은 하지 마세요."
                    )
                },
                {
                    "role": "user",
                    "content": f"대화:\n{dialogue}\n\n주제:"
                }
            ],
            temperature=0.0,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 10)
            else:
                return ""  # 실패 시 빈 topic

### 2-2. 역할 추론 함수
- #Person1#, #Person2# 등 화자의 역할/직업/이름 추론
> 이름(Frank, Jimmy) > 직업(의사, 환자) > 관계(친구) 우선순위로 추론

In [40]:
def infer_roles(dialogue, max_retries=3):
    """역할 추론 - 대화에서 각 화자의 역할/직업/이름 추론"""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="solar-1-mini-chat",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "대화를 읽고 각 화자의 역할을 추론하세요.\n"
                            "추론 우선순위:\n"
                            "  1순위: 대화에서 직접 언급된 이름 (예: Frank, Jimmy)\n"
                            "  2순위: 직업/역할 (예: 의사, 환자, 선생님)\n"
                            "  3순위: 관계 (예: 친구, 직장동료)\n"
                            "  확실하지 않으면: 알 수 없음\n"
                            "반드시 JSON 형식으로만 답하세요.\n"
                            "예: {\"#Person1#\": \"Frank(직장동료)\", \"#Person2#\": \"의사\"}"
                        )
                    },
                    {
                        "role": "user",
                        "content": f"대화:\n{dialogue}\n\n역할:"
                    }
                ],
                temperature=0.0,
            )
            text = response.choices[0].message.content.strip()
            match = re.search(r'\{.*\}', text, re.DOTALL)
            if match:
                return json.loads(match.group())
            return {}

        except Exception as e:
            print(f"  ⚠️ infer_roles 시도 {attempt+1}/{max_retries} 실패: {type(e).__name__}")
            if attempt < max_retries - 1:
                wait = (attempt + 1) * 10
                print(f"  {wait}초 후 재시도...")
                time.sleep(wait)
            else:
                print(f"  ❌ infer_roles 최종 실패 - 빈 역할 반환")
                return {}


### 2-3. topic 기반 few-shot 선택 함수
> 완전 일치 대신 문자 n-gram 유사도로 가장 비슷한 topic의 train 샘플 선택

In [41]:
def get_few_shots_by_topic(predicted_topic, n=3):
    """TF-IDF 유사도로 가장 유사한 train 샘플 선택"""
    try:
        pred_vec = vectorizer.transform([predicted_topic])
        sims = cosine_similarity(pred_vec, topic_matrix).flatten()
        top_indices = sims.argsort()[::-1][:n*2]  # 여유있게 2배 추출

        # 유사도 0.1 이상인 것만 사용
        valid = [i for i in top_indices if sims[i] >= 0.1]
        if valid:
            return train_df.iloc[valid[:n]].to_dict('records')
        else:
            return train_df.sample(n, random_state=42).to_dict('records')
    except:
        return train_df.sample(n, random_state=42).to_dict('records')

### 2-4. build_prompt & 요약 함수
- 한국어 프롬프트
- topic 기반 few-shot 예시 3개
- 역할 추론 힌트 포함
- temperature=0.2, top_p=0.3
> 역할 정보를 요약에 반드시 반영하도록 강조

In [ ]:
def build_prompt(dialogue, predicted_topic=None, roles=None):
    """topic + 역할 추론 + few-shot 반영한 최종 프롬프트"""
    system_prompt = (
        "당신은 한국어 대화 요약 전문가입니다. "
        "주어진 대화의 핵심 내용을 한국어 문어체로 반드시 1문장으로 요약하세요. "
        "#Person1#, #Person2# 등 화자 태그를 절대 바꾸지말고 반드시 그대로 유지하세요."
        "이름(Jimmy, Karen 등)이나 역할(아내, 남편, 의사 등)로 절대 대체하지 마세요."
        "화자 역할 정보는 참고만 하고 요약문에는 반드시 태그를 사용하세요."
        "불필요한 세부사항은 생략하고 핵심 행동과 결과만 포함하세요."
        # "절대로 태그를 이름이나 역할로 바꾸지 마세요."
        # "화자의 이름이나 역할이 파악된 경우 태그 대신 실제 이름/역할을 사용하세요."  # ✅ 추가
    )

    # 역할 힌트 추가
    role_hint = ""
    if roles:
        valid_roles = {k: v for k, v in roles.items() if v != "알 수 없음"}
        if valid_roles:
            # role_hint = "화자 역할 정보 (요약에 반드시 반영하세요):\n"
            role_hint = "참고 정보(요약에서 태그는 그대로 유지할 것): \n"
            # role_hint = "※ 참고(요약에 쓰지 말 것):\n"
            for person, role in valid_roles.items():
                role_hint += f"  {person} = {role}\n"
            role_hint += "\n"

    # few-shot 예시 선택
    if predicted_topic:
        few_shots = get_few_shots_by_topic(predicted_topic, n=3)
    else:
        few_shots = train_df.sample(3, random_state=42).to_dict('records')

    examples = ''
    for shot in few_shots:
        examples += f"대화:\n{shot['dialogue']}\n요약: {shot['summary']}\n\n"

    user_prompt = (
        f"아래 예시를 참고하여 마지막 대화를 요약하세요.\n\n"
        f"{examples}"
        f"{role_hint}"   # ✅ 참고만 하도록
        f"대화:\n{dialogue}\n"
        f"위 대화를 #Person 태그를 그대로 유지하여 1문장으로 요약하세요:\n"  # ✅ 마지막 강조
        f"요약:"
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt}
    ]


### (선택) parameter 변경하기
- Solar Chat API를 사용할 때, parameter를 변경하여, 다양한 결과를 얻을 수 있습니다.
- Parameter에 대한 자세한 설명은 [여기](https://developers.upstage.ai/docs/apis/chat#request-body)를 참고해주세요.

In [43]:
# Solar Chat API를 활용해 Summarization을 수행하는 함수를 정의합니다.
def summarization(dialogue, predicted_topic=None, roles=None, max_retries=3):
    """Solar API로 요약 생성 - 타임아웃/서버 에러 시 자동 재시도"""
    for attempt in range(max_retries):
        try:
            summary = client.chat.completions.create(
                model="solar-1-mini-chat",
                messages=build_prompt(dialogue, predicted_topic, roles),
                temperature=0.0, # 더 일관된 짧은 요약
                top_p=0.1,
            )
            return summary.choices[0].message.content.strip()

        except Exception as e:
            print(f"  ⚠️ summarization 시도 {attempt+1}/{max_retries} 실패: {type(e).__name__}")
            if attempt < max_retries - 1:
                wait = (attempt + 1) * 10  # 10초, 20초, 30초 점점 늘려서 재시도
                print(f"  {wait}초 후 재시도...")
                time.sleep(wait)
            else:
                print(f"  ❌ 최종 실패 - 빈 문자열 반환")
                return "" # 실패 시 빈 문자열로 대체


## 3. 샘플 테스트
> train 데이터 3개로 역할 추론 & 요약 품질 확인

In [44]:
# Train data 중 처음 3개의 대화를 요약합니다.
def test_on_train_data(num_samples=3):
    samples = train_df.sample(num_samples, random_state=42)
    for _, row in samples.iterrows():
        dialogue = row['dialogue']
        gold     = row['summary']

        predicted_topic = predict_topic(dialogue)
        roles           = infer_roles(dialogue)
        pred            = summarization(dialogue, predicted_topic, roles)
        score           = compute_metrics(pred, gold)

        print(f"Dialogue:\n{dialogue}\n")
        print(f"예측 Topic : {predicted_topic}")
        print(f'[실제 topic] {row.get("topic", "없음")}')
        print(f"추론 역할  : {roles}")
        print(f"예측(Pred Summary): {pred}")
        print(f"정답(Gold Summary): {gold}")
        print(f'[ROUGE] R1={score["rouge-1"]:.3f} R2={score["rouge-2"]:.3f} RL={score["rouge-l"]:.3f} AVG={score["avg"]:.3f}')
        print("="*80)

In [45]:
if __name__ == "__main__":
    test_on_train_data()

Dialogue:
#Person1#: 안녕하세요, 잭 있나요?
#Person2#: 전데요.
#Person1#: 잭! 나 로즈야.
#Person2#: 안녕, 로즈. 어떻게 지내?
#Person1#: 잘 지내, 고마워. 이번 주 토요일 저녁에 친구들 몇 명 초대했어. 너도 같이 올 수 있는지 궁금해서.
#Person2#: 좋네. 몇 시에 가면 될까?
#Person1#: 여섯 시 괜찮아?

예측 Topic : 생일 파티 초대
[실제 topic] 저녁 초대
추론 역할  : {'#Person1#': '로즈(친구)', '#Person2#': '잭(친구)'}
예측(Pred Summary): 로즈가 잭에게 이번 주 토요일 저녁에 열리는 친구 모임에 초대하고, 잭은 여섯 시에 참석하겠다고 응답합니다.
정답(Gold Summary): 로즈는 잭에게 전화하여 이번 주 토요일 저녁 식사에 초대한다.
[ROUGE] R1=0.333 R2=0.182 RL=0.333 AVG=0.283
Dialogue:
#Person1#: 저기요, 면접 보러 갈 때 뭐 입어야 할까요?
#Person2#: 정장에 넥타이를 매는 게 좋을 것 같아요.
#Person1#: 면접 중에 긴장할까 봐 걱정이에요.
#Person2#: 걱정 마세요. 그냥 최선을 다해서 자신을 잘 표현하세요.

예측 Topic : 면접 준비, 자신감 부여
[실제 topic] 면접 준비
추론 역할  : {'#Person1#': '알 수 없음', '#Person2#': '알 수 없음'}
예측(Pred Summary): #Person1#이 면접 복장과 긴장에 대해 걱정하자, #Person2#는 정장과 넥타이를 착용하고 자신을 최선을 다해 표현하라고 조언합니다.
정답(Gold Summary): #Person2#는 #Person1#에게 면접에서 정장과 넥타이를 착용하고 자신을 잘 표현하라고 조언합니다.
[ROUGE] R1=0.560 R2=0.348 RL=0.560 AVG=0.489
Dialogue:
#Person1#: 존, 동기부여에 대해 몇 가지

## 4. Validation 성능 평가
> **목적**: 리더보드 제출 전 로컬 ROUGE 점수 확인 + 어디서 틀리는지 파악

- `num_samples=50` → 빠른 확인 (약 10~15분)
- `num_samples=-1` → 전체 499개 (약 1시간)

In [ ]:
# Validation data의 대화를 요약하고, 점수를 측정합니다.
def validate(num_samples=50):
    """dev.csv로 ROUGE 평가 + 실패 케이스 저장"""

    import mecab
    m = mecab.MeCab()

    def mecab_tokenize(text):
        return ''.join([tok for tok, _ in m.pos(text)])
    val_samples = val_df[:num_samples] if num_samples > 0 else val_df

    results_list = []
    start_time = time.time()

    for idx, row in tqdm(val_samples.iterrows(), total=len(val_samples)):
        dialogue = row['dialogue']
        gold     = row['summary']

        predicted_topic = predict_topic(dialogue)
        roles           = infer_roles(dialogue)
        pred            = summarization(dialogue, predicted_topic, roles)
        score           = compute_metrics(pred, gold)

        # MeCab 형태소 분석 후 ROUGE 계산
        pred_tok = mecab_tokenize(pred)
        gold_tok = mecab_tokenize(gold)

        try:
            score = rouge.get_scores(pred_tok, gold_tok)[0]
            r1 = score['rouge-1']['f']
            r2 = score['rouge-2']['f']
            rl = score['rouge-l']['f']
        except:
            r1 = r2 = rl = 0.0

        results_list.append({
            'fname'          : row['fname'],
            'dialogue'       : dialogue,
            'gold'           : gold,
            'pred'           : pred,
            'predicted_topic': predicted_topic,
            'roles'          : str(roles),
            'rouge1'         : score['rouge-1'],
            'rouge2'         : score['rouge-2'],
            'rougeL'         : score['rouge-l'],
            'avg'            : score['avg'],
        })

        # RPM 제한 방지 (샘플당 API 3번 호출 → 33개마다 대기)
        if (idx + 1) % 33 == 0:
            elapsed = time.time() - start_time
            if elapsed < 60:
                wait = 60 - elapsed + 5
                print(f"RPM 제한 방지 대기: {wait:.0f}초")
                time.sleep(wait)
            start_time = time.time()

    result_df = pd.DataFrame(results_list)

    # 점수 출력
    print('\n' + '='*60)
    print('Validation ROUGE 점수')
    print('='*60)
    print(f'  ROUGE-1 : {result_df["rouge1"].mean():.4f}')
    print(f'  ROUGE-2 : {result_df["rouge2"].mean():.4f}')
    print(f'  ROUGE-L : {result_df["rougeL"].mean():.4f}')
    print(f'  평균    : {result_df["avg"].mean():.4f}')
    print('='*60)

    # 저장
    os.makedirs(RESULT_PATH, exist_ok=True)
    save_path = os.path.join(RESULT_PATH, 'solar_val_results.csv')
    result_df.to_csv(save_path, index=False)
    print(f'\n결과 저장: {save_path}')

    return result_df

In [47]:
if __name__ == "__main__":
     val_result = validate(10)  # 100 -> 50개의 validation sample에 대한 요약을 수행합니다.

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:19<00:00,  1.98s/it]


Validation ROUGE 점수
  ROUGE-1 : 0.2606
  ROUGE-2 : 0.0734
  ROUGE-L : 0.2525
  평균    : 0.1955

결과 저장: /data/ephemeral/home/code/prediction/solar_val_results.csv


In [48]:
# 실패 케이스 분석 - validate() 실행 후 이 셀 실행
def analyze_failures(result_df, top_n=10):
    """ROUGE 낮은 케이스 분석"""
    sorted_df = result_df.sort_values('avg')

    print(f'전체 평균 ROUGE: {result_df["avg"].mean():.4f}')
    print(f'하위 25%  ROUGE: {result_df["avg"].quantile(0.25):.4f}')
    print(f'상위 25%  ROUGE: {result_df["avg"].quantile(0.75):.4f}')

    print(f'\n{"-"*70}')
    print(f'하위 {top_n}개 실패 케이스 분석')
    print('-'*70)

    for _, row in sorted_df.head(top_n).iterrows():
        print(f'\n[{row["fname"]}] AVG={row["avg"]:.3f} | topic={row["predicted_topic"]}')
        print(f'역할: {row["roles"]}')
        print(f'정답: {row["gold"]}')
        print(f'예측: {row["pred"]}')
        print('-'*70)

    # 패턴 분석
    print('\n[패턴 분석]')
    pred_lens = result_df['pred'].apply(lambda x: len(str(x).split()))
    gold_lens = result_df['gold'].apply(lambda x: len(str(x).split()))
    print(f'  예측 평균 길이: {pred_lens.mean():.1f}단어')
    print(f'  정답 평균 길이: {gold_lens.mean():.1f}단어')

    # 역할 추론 성공률
    has_role = result_df['roles'].apply(lambda x: x != '{}' and '알 수 없음' not in str(x))
    print(f'  역할 추론 성공: {has_role.sum()}/{len(result_df)} ({has_role.mean()*100:.1f}%)')

    # 정답에 Person 태그 vs 이름 사용 비율
    gold_has_name = result_df['gold'].apply(
        lambda x: bool(re.search(r'[A-Z][a-z]+', str(x)))
    )
    pred_has_name = result_df['pred'].apply(
        lambda x: bool(re.search(r'[A-Z][a-z]+', str(x)))
    )
    print(f'  정답에 이름 사용: {gold_has_name.mean()*100:.1f}%')
    print(f'  예측에 이름 사용: {pred_has_name.mean()*100:.1f}%')

In [49]:
if __name__ == '__main__':
    analyze_failures(val_result, top_n=10)

전체 평균 ROUGE: 0.1955
하위 25%  ROUGE: 0.1237
상위 25%  ROUGE: 0.2258

----------------------------------------------------------------------
하위 10개 실패 케이스 분석
----------------------------------------------------------------------

[dev_7] AVG=0.085 | topic=해변 여행과 여가 활동
역할: {'#Person1#': '알 수 없음', '#Person2#': 'Karen(직장동료)'}
정답: #Person1#은 Karen에게 해변에서 주말을 어떻게 보냈는지, 어디에서 머물렀는지 묻습니다. #Person1#은 그 경험이 편안해 보이고 본인도 해변에 가고 싶다고 말합니다.
예측: #Person1#은 #Person2#가 주말에 해변에 다녀와 태닝한 것을 알아채고, #Person2#는 부모님 친구분의 초대로 해변에서 조깅과 배구를 즐기며 휴식을 취했으나 물의 차가움으로 수영은 많이 하지 못했다고 설명하며 #Person1#도 휴식이 필요하다고 제안합니다.
----------------------------------------------------------------------

[dev_4] AVG=0.099 | topic=학교 생활, 영화 관람 계획
역할: {'#Person1#': '알 수 없음', '#Person2#': '알 수 없음'}
정답: #Person1#은 오늘 학교에 가지 않았고, #Person2#는 내일 학교 대신 영화관에 가고 싶어합니다.
예측: #Person2#는 #Person1#에게 이번 주말에 함께 영화를 보러 가자고 제안했으나 #Person1#은 거절하고, #Person2#는 결국 혼자 영화를 보러 가기로 결정했습니다.
----------------------------------------------------------------------

[dev_0] 

## 5. Test 추론 및 제출 파일 생성
> test.csv 499개 전체 추론 (약 30~50분)
- topic 예측 + 역할 추론 + few-shot 반영한 최종 inference

In [50]:
def inference():
    """test.csv 전체 추론 -> output_solar.csv 저장"""
    test_data  = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))
    summaries = []
    start_time = time.time()

    for idx, row in tqdm(test_data.iterrows(), total=len(test_data)):
        dialogue = row['dialogue']
        
        # STEP 1. topic 예측
        predicted_topic = predict_topic(dialogue)

        # STEP 2. 역할 추론
        roles = infer_roles(dialogue)

        # STEP 3. 요약 생성
        summary = summarization(dialogue, predicted_topic, roles)
        summaries.append(summary)
        
        # Rate limit 방지를 위해 1분 동안 최대 100개의 요청을 보내도록 합니다.
        # RPM 제한 방지 (샘플당 API 3번 호출 → 33개마다 대기)
        if (idx + 1) % 20 == 0: # 33-> 20으로 줄임
            elapsed = time.time() - start_time

            if elapsed < 60:
                wait = 60 - elapsed + 10 # 5-> 10초 여유 추가
                print(f"RPM 제한 방지 대기: {wait:.0f}초")
                time.sleep(wait)

            start_time = time.time()
    
    output = pd.DataFrame(
        {
            "fname": test_df['fname'],
            "summary" : summaries,
        }
    )
    
    os.makedirs(RESULT_PATH, exist_ok=True)
    save_path = os.path.join(RESULT_PATH, "output_solar.csv")
    output.to_csv(save_path, index=False)
    print(f"\n저장 완료: {save_path}")
    print(f'총 {len(output)}개 | 고유 요약: {output["summary"].nunique()}개')

    return output

In [51]:
if __name__ == "__main__":
    output = inference()

  0%|          | 0/499 [00:00<?, ?it/s]

  4%|▍         | 19/499 [00:34<14:32,  1.82s/it]

RPM 제한 방지 대기: 34초


  8%|▊         | 39/499 [01:48<15:07,  1.97s/it]  

RPM 제한 방지 대기: 29초


...

RPM 제한 방지 대기: 32초


 96%|█████████▌| 479/499 [27:30<00:39,  1.99s/it]

RPM 제한 방지 대기: 29초


100%|██████████| 499/499 [28:40<00:00,  3.45s/it]


저장 완료: /data/ephemeral/home/code/prediction/output_solar.csv
총 499개 | 고유 요약: 499개


- 중간에 끊긴 경우 이어서 돌리는 방법
> 처음부터 다시 돌리면 API 낭비! 이미 저장된 결과 이어서 돌릴 수 있어요.

In [52]:
def inference_resume():
    """중간에 끊겼을 때 이어서 추론"""
    test_df = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))
    
    # 이미 완료된 결과 불러오기
    save_path = os.path.join(RESULT_PATH, 'output_solar.csv')
    if os.path.exists(save_path):
        done_df   = pd.read_csv(save_path)
        done_fnames = set(done_df['fname'].tolist())
        print(f'이미 완료: {len(done_fnames)}개 / 전체: {len(test_df)}개')
    else:
        done_df     = pd.DataFrame(columns=['fname', 'summary'])
        done_fnames = set()

    summaries  = done_df.to_dict('records')
    start_time = time.time()

    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        if row['fname'] in done_fnames:
            continue  # ✅ 이미 완료된 건 건너뜀

        dialogue        = row['dialogue']
        predicted_topic = predict_topic(dialogue)
        roles           = infer_roles(dialogue)
        summary         = summarization(dialogue, predicted_topic, roles)
        summaries.append({'fname': row['fname'], 'summary': summary})

        if (idx + 1) % 20 == 0:
            elapsed = time.time() - start_time
            if elapsed < 60:
                time.sleep(60 - elapsed + 10)
            start_time = time.time()

            # ✅ 20개마다 중간 저장
            pd.DataFrame(summaries).to_csv(save_path, index=False)
            print(f'중간 저장 완료: {len(summaries)}개')

    output = pd.DataFrame(summaries)
    output.to_csv(save_path, index=False)
    print(f'완료! {len(output)}개 | 고유 요약: {output["summary"].nunique()}개')
    return output


if __name__ == '__main__':
    output = inference_resume()


이미 완료: 499개 / 전체: 499개


100%|██████████| 499/499 [00:00<00:00, 19879.16it/s]

완료! 499개 | 고유 요약: 499개


## 6. 결과확인

In [53]:
# test 추론 결과 확인
output = pd.read_csv(os.path.join(RESULT_PATH, 'output_solar.csv'))
print(f"총 행 수 : {len(output)}")
print(f"고유 요약 수 : {output['summary'].nunique()}")
print(output.head(10).to_string())

총 행 수 : 499
고유 요약 수 : 499
    fname                                                                                                                    summary
0  test_0              #Person1#은 #Person2#에게 즉시 메시지 프로그램 사용 금지에 관한 사내 메모 받아쓰기를 요청하고, #Person2#는 정책의 적용 범위에 대해 질문한 후 메모 작성을 계속 진행한다.
1  test_1                             #Person1#과 #Person2#는 교통체증으로 인한 통근의 어려움을 토로하며, 대중교통이나 자전거 이용 등 환경과 스트레스를 줄일 수 있는 대안을 모색하기로 했다.
2  test_2                  #Person1#은 #Person2#에게 Masha와 Hero가 두 달간 별거 끝에 이혼 신청을 했으며, Masha가 양육권을 가지기로 했고 모든 절차가 원만하게 진행되고 있다고 전합니다.
3  test_3                                                     여자친구인 #Person1#이 Brian인 #Person2#의 생일을 축하하며 선물을 전달하고 함께 춤을 추며 파티를 즐긴다.
4  test_4                     #Person1#과 #Person2#는 올림픽 스타디움에 있으며, 스타디움의 규모와 완공 시기, 좌석 수 등에 대해 이야기하고 외국인 방문객을 위한 영어 표지판에 대해서도 언급합니다.
5  test_5       #Person1#은 회사에 사직하고 자신의 사업을 시작하려 하지만 #Person2#는 사업 계획서 작성의 복잡성과 시장 분석, 금융 분석의 중요성을 설명하며 #Person1#의 결정에 대한 신중함을 일깨운다.
6  test_6                                  

In [54]:
# val 평가 결과 비교 (validate() 실행 후)
try:
    val_result = pd.read_csv(os.path.join(RESULT_PATH, 'solar_val_results.csv'))
    print(f'Validation 샘플 수: {len(val_result)}')
    print(f'ROUGE-1 평균: {val_result["rouge1"].mean():.4f}')
    print(f'ROUGE-2 평균: {val_result["rouge2"].mean():.4f}')
    print(f'ROUGE-L 평균: {val_result["rougeL"].mean():.4f}')
    print(f'AVG     평균: {val_result["avg"].mean():.4f}')
    analyze_failures(val_result, top_n=5)
except FileNotFoundError:
    print('validate() 먼저 실행하세요!')

Validation 샘플 수: 10
ROUGE-1 평균: 0.2606
ROUGE-2 평균: 0.0734
ROUGE-L 평균: 0.2525
AVG     평균: 0.1955
전체 평균 ROUGE: 0.1955
하위 25%  ROUGE: 0.1237
상위 25%  ROUGE: 0.2258

----------------------------------------------------------------------
하위 5개 실패 케이스 분석
----------------------------------------------------------------------

[dev_7] AVG=0.085 | topic=해변 여행과 여가 활동
역할: {'#Person1#': '알 수 없음', '#Person2#': 'Karen(직장동료)'}
정답: #Person1#은 Karen에게 해변에서 주말을 어떻게 보냈는지, 어디에서 머물렀는지 묻습니다. #Person1#은 그 경험이 편안해 보이고 본인도 해변에 가고 싶다고 말합니다.
예측: #Person1#은 #Person2#가 주말에 해변에 다녀와 태닝한 것을 알아채고, #Person2#는 부모님 친구분의 초대로 해변에서 조깅과 배구를 즐기며 휴식을 취했으나 물의 차가움으로 수영은 많이 하지 못했다고 설명하며 #Person1#도 휴식이 필요하다고 제안합니다.
----------------------------------------------------------------------

[dev_4] AVG=0.099 | topic=학교 생활, 영화 관람 계획
역할: {'#Person1#': '알 수 없음', '#Person2#': '알 수 없음'}
정답: #Person1#은 오늘 학교에 가지 않았고, #Person2#는 내일 학교 대신 영화관에 가고 싶어합니다.
예측: #Person2#는 #Person1#에게 이번 주말에 함께 영화를 보러 가자고 제안했으나 #Person1#은 거절하고, #Person2#는 결국 혼자 영화를 

---